# Extract Stances from Every Legislator Turn, in Shards

| Item | Description |
|---|---|
| Goal | Run the stance extraction of T2.2.2 over all 220,160 non-empty legislator turns: one call per turn window, five shards in parallel, each shard one array task on one GPU. The output is the machine-checked records that T2.2.4 assembles into one table. |
| Input | `member_turns.parquet` from T2.2.1; the model Qwen/Qwen3.5-4B from the project cache. |
| Output | In the T2.2 Interim folder: `shards/records_00.parquet` to `records_04.parquet`, one row per turn with its records as JSON, and `status/extract_00.jsonl` to `extract_04.jsonl`, one line per call. |
| Variables | • `SHARD`, `N_SHARDS`: read from the job environment; shard i takes every fifth turn starting at i, on the stored turn order<br>• `records_json`: the merged records of the turn<br>• `errors`: named parse errors, including `hit_token_cap` |
| Run | Ran 2026-08-22 as a five-task GPU array (job 11261709), 7.3 to 9.6 hours per shard on A40s; the outputs shown are from the executed copy of shard 0, and Step 11 reads the logs of all five shards. Not rerun: the records are the production data and sampling would change them. The job script is `J - Jobs/run_T2.2.3_extract_stances.sbatch`. |


## Step 1. Import packages


In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## Step 2. Define file locations


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction")
NOTEBOOK_NAME = "T2.2.3 - Production - Extract Stances from Every Turn in Shards"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SHARD_FOLDER = INTERIM_FOLDER / "shards"
STATUS_FOLDER = INTERIM_FOLDER / "status"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 3. Define the prompt

The wording of T2.2.2, unchanged.


In [ ]:
PROMPT_VERSION = "stance-extraction-0.3-dev"  # stamped on every logged call; the production run kept this recorded name
WINDOW_SENTENCES = 40  # a longer turn is sent in consecutive windows of this size, with turn-global sentence numbers
STANCES = ("support", "concern")

SYSTEM = ("You are an expert reader of United States congressional hearing transcripts. "
          "You report only what the text states.")

USER = """Below is one speaking turn by a member of Congress, split into numbered sentences.

Task: find every sentence in which the member takes a directional position on a policy object, and return one record per position.

A record has three fields:
- "sentences": the sentence number(s) expressing the position (1-3 consecutive numbers);
- "topic": the policy object, as a short and specific neutral noun phrase (the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it);
- "stance": "support" if the member endorses, advocates, or urges adoption, funding, or expansion of the topic; "concern" if the member expresses worry, criticism, or objection toward the topic, or urges its restriction, delay, or scrutiny.

The stance is toward the topic exactly as you phrase it. Before returning a record, check that it reads sensibly as "the member supports [topic]" or "the member is concerned about [topic]". If the topic is a problem or harm, the stance toward it is "concern", never "support".

The topic must be the object the member takes a position ON. Entities mentioned only as examples, comparisons, or asides are not topics, and a bare number or amount is never a topic. If several consecutive sentences express the same position on the same topic, return one record covering them, not one record per sentence.

Example: "Youth vaping is an epidemic we cannot ignore." -> {{"sentences": [n], "topic": "youth e-cigarette use", "stance": "concern"}}
Example: "This bill funds the research our labs badly need." -> {{"sentences": [n], "topic": "research funding bill", "stance": "support"}}

Do not return records for: procedure (recognitions, time, the record), courtesies, introductions, logistics, questions asked without an expressed position, or factual statements without evaluation. If no sentence takes a directional position, return an empty list. When in doubt, do not extract.

--- TURN (Speaker: {speaker}) ---
{numbered}
--- END OF TURN ---

Return only a JSON list of records. Your very first output character must be [."""

# Directional words that must not appear inside a topic phrase; direction belongs in the stance field.
# The lint flags a phrase for review. It does not rewrite it.
DIRECTIONAL_LINT = re.compile(
    r"\b(?:ban(?:ning)?|block(?:ing)?|stop(?:ping)?|halt(?:ing)?|"
    r"oppos(?:e|ing|ition)|support(?:ing)?|defend(?:ing)?|protect(?:ing|ion)?|"
    r"against|anti|pro|promot(?:e|ing|ion)|expand(?:ing)?|restrict(?:ing|ion)?|"
    r"repeal(?:ing)?|eliminat(?:e|ing|ion))\b",
    re.IGNORECASE,
)


def lint_topic(phrase):
    """The directional word a topic phrase contains, or None if clean."""
    match = DIRECTIONAL_LINT.search(phrase or "")
    return match.group(0) if match else None


## Step 4. Define the windows and the parser

As in T2.2.2.


In [ ]:
def render_user(sentences, speaker, start=1):
    """The extraction prompt for one turn, or one window of a long turn."""
    return USER.format(speaker=speaker, numbered=format_numbered(sentences, start=start))


def windows(sentences, size=WINDOW_SENTENCES):
    """Yield (first sentence number, sentences) for each window of a turn; a short turn is one window."""
    for offset in range(0, max(len(sentences), 1), size):
        yield offset + 1, sentences[offset:offset + size]


def parse_records(output, n_sentences, start=1):
    """Validate one model output into records and a list of named errors.

    Every check that code can make is made here: JSON shape, field presence, stance vocabulary,
    sentence indices within the window (at most 3, consecutive), and the directional lint on the topic.
    A record that fails a structural check is dropped and counted; a lint hit keeps the record with a flag.
    """
    errors = []
    text = output.strip()
    if text.startswith("```"):  # a markdown fence is tolerated but counted
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        return [], errors + ["json_parse"]
    if not isinstance(raw, list):
        return [], errors + ["not_a_list"]
    records = []
    for item in raw:
        if not isinstance(item, dict):
            errors.append("record_not_object")
            continue
        topic, stance, sent = item.get("topic"), item.get("stance"), item.get("sentences")
        if isinstance(sent, int):
            sent = [sent]
        if not (isinstance(topic, str) and topic.strip()):
            errors.append("missing_topic")
            continue
        if stance not in STANCES:
            errors.append("bad_stance")
            continue
        if not (isinstance(sent, list) and sent and all(isinstance(i, int) for i in sent)):
            errors.append("bad_sentences_field")
            continue
        sent = sorted(set(sent))
        if sent[0] < start or sent[-1] > start + n_sentences - 1:
            errors.append("index_out_of_range")
            continue
        if len(sent) > 3 or sent[-1] - sent[0] != len(sent) - 1:
            errors.append("indices_not_consecutive")
            continue
        records.append({"sentences": sent, "topic": topic.strip(), "stance": stance, "topic_lint": lint_topic(topic)})
    return records, errors


def merge_adjacent_records(records):
    """Merge records with the same topic and stance whose sentences touch or sit within two sentences; returns the list and the merge count."""
    if not records:
        return records, 0
    ordered = sorted(records, key=lambda r: (r["topic"].strip().lower(), r["stance"], r["sentences"][0]))
    merged, n_merged = [], 0
    for record in ordered:
        if merged:
            prev = merged[-1]
            if (prev["topic"].strip().lower() == record["topic"].strip().lower() and prev["stance"] == record["stance"]
                    and record["sentences"][0] - prev["sentences"][-1] <= 2):
                prev["sentences"] = sorted(set(prev["sentences"] + record["sentences"]))
                n_merged += 1
                continue
        merged.append(dict(record))
    merged.sort(key=lambda r: r["sentences"][0])
    return merged, n_merged


## Step 5. Define the model call and the log


In [ ]:
MAX_NEW_TOKENS = 1024

GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, the same setting as T1


def render(tokenizer, system, user):
    """Apply the model's chat template, with the thinking mode off where the template offers it."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, system, user, max_new_tokens):
    """One prompt, one completion, fresh context. Also returns whether the completion ran into the token cap."""
    prompt = render(tokenizer, system, user)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = ids["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **GENERATION)
    generated = out[0][n_in:]
    hit_cap = len(generated) >= max_new_tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), hit_cap


class CallLog:
    """One JSON line per model call, written as it happens: stage, level, position, a hash of the exact input, token counts, time, and the full output. The same format as T1."""

    def __init__(self, path, document_id, model_name, prompt_version):
        self.path = Path(path) if path else None
        self.document_id = document_id
        self.model_name = model_name
        self.prompt_version = prompt_version
        self.records = []
        if self.path:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, stage, level, index, n_inputs, prompt_text, output, tokenizer, seconds):
        record = {
            "document_id": self.document_id, "model": self.model_name, "prompt_version": self.prompt_version,
            "stage": stage, "level": level, "index": index, "n_inputs": n_inputs,
            "input_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
            "input_chars": len(prompt_text),
            "input_tokens": len(tokenizer.encode(prompt_text, add_special_tokens=False)),
            "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
            "output_chars": len(output), "seconds": round(seconds, 2),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "output": output,
        }
        self.records.append(record)
        if self.path:
            with self.path.open("a") as log_file:
                log_file.write(json.dumps(record) + "\n")
                log_file.flush()


## Step 6. Read the run settings from the environment

Every setting comes from the job script, so one notebook serves all five tasks and each records the shard it ran.


In [1]:
MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
SHARD = int(os.environ.get("SHARD", 0))
N_SHARDS = int(os.environ.get("N_SHARDS", 5))
SHARD_FOLDER.mkdir(parents=True, exist_ok=True)
STATUS_FOLDER.mkdir(parents=True, exist_ok=True)
SHARD_FILE = SHARD_FOLDER / f"records_{SHARD:02d}.parquet"
LOG_FILE = STATUS_FOLDER / f"extract_{SHARD:02d}.jsonl"

print(f"model   : {MODEL}")
print(f"prompt  : {PROMPT_VERSION}")
print(f"shard   : {SHARD} of {N_SHARDS}")
print(f"window  : {WINDOW_SENTENCES} sentences, cap {MAX_NEW_TOKENS}")
print(f"sampling: {GENERATION}")
print(f"gpu     : {torch.cuda.get_device_name(0)}")


model   : Qwen/Qwen3.5-4B
prompt  : stance-extraction-0.3-dev
shard   : 0 of 5
window  : 40 sentences, cap 1024
sampling: {'do_sample': True, 'temperature': 0.1}
gpu     : NVIDIA A40


## Step 7. Select this shard's turns

Turns emptied by the cleaning rules are skipped. Turns already in the shard file are skipped too, so a resubmitted task continues where it stopped.


In [2]:
turns = pd.read_parquet(PROCESSED_FOLDER / "member_turns.parquet")
work = turns.iloc[SHARD::N_SHARDS]
work = work.loc[work["n_sentences"] > 0].reset_index(drop=True)

rows = []
if SHARD_FILE.exists():
    rows = pd.read_parquet(SHARD_FILE).to_dict("records")
done = {(r["packageId"], r["turn_idx"]) for r in rows}
todo = work.loc[[(p, t) not in done for p, t in zip(work["packageId"], work["turn_idx"])]].reset_index(drop=True)

print(f"shard turns : {len(work):,} ({work['n_sentences'].sum():,} sentences)")
print(f"  done      : {len(done):,}")
print(f"  to extract: {len(todo):,}")


shard turns : 44,020 (196,553 sentences)
  done      : 0
  to extract: 44,020


## Step 8. Load the model


In [3]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s, {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")


loaded in 12s — 4.21B params


## Step 9. Extract

Per turn: seed the sampler from the turn key, one call per window, parse, merge adjacent duplicates. The shard file is rewritten every 200 turns and the log is flushed per call, so nothing that happened exists only in memory.


In [4]:
started = time.time()
n_calls_session = 0
for n, row in enumerate(todo.itertuples(), 1):
    key = f"{row.packageId}:{row.turn_idx}"
    torch.manual_seed(int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big"))
    speaker = f"{row.honorific or ''} {row.surname or ''}".strip() or "Unidentified"
    log = CallLog(LOG_FILE, key, MODEL, prompt_version=PROMPT_VERSION)

    records, errors = [], []
    for w_index, (w_start, w_sents) in enumerate(windows(list(row.sentences))):
        user = render_user(w_sents, speaker, start=w_start)
        t0 = time.time()
        output, hit_cap = generate(model, tokenizer, SYSTEM, user, max_new_tokens=MAX_NEW_TOKENS)
        log.write("extract", 0, w_index, 1, user, output, tokenizer, time.time() - t0)
        n_calls_session += 1
        w_records, w_errors = parse_records(output, len(w_sents), start=w_start)
        if hit_cap:
            w_errors.append("hit_token_cap")
        records.extend(w_records)
        errors.extend(w_errors)
    records, n_merged = merge_adjacent_records(records)

    rows.append({
        "packageId": row.packageId, "turn_idx": int(row.turn_idx), "year": int(row.year), "chamber": row.chamber,
        "bioGuideId": row.bioGuideId, "surname": row.surname, "honorific": row.honorific, "is_chair": bool(row.is_chair),
        "n_sentences": int(row.n_sentences), "prompt_version": PROMPT_VERSION, "model": MODEL,
        "n_records": len(records), "n_merged": n_merged, "records_json": json.dumps(records), "errors": ";".join(errors),
    })
    if n % 200 == 0 or n == len(todo):
        pd.DataFrame(rows).to_parquet(SHARD_FILE, compression="zstd", index=False)
        rate = n / (time.time() - started)
        print(f"  {n:>6,}/{len(todo):,}  {rate:.2f} turns/s  eta {(len(todo) - n) / rate / 3600:.1f} h")

pd.DataFrame(rows).to_parquet(SHARD_FILE, compression="zstd", index=False)
print(f"\nshard {SHARD}: {len(rows):,} turns, {n_calls_session:,} calls this session, {(time.time() - started) / 3600:.2f} h")


     200/44,020  1.63 turns/s  eta 7.4 h


     400/44,020  1.45 turns/s  eta 8.4 h


     600/44,020  1.28 turns/s  eta 9.4 h


     800/44,020  1.23 turns/s  eta 9.8 h


   1,000/44,020  1.21 turns/s  eta 9.9 h


   1,200/44,020  1.18 turns/s  eta 10.1 h


... (211 outputs left out)

  43,600/44,020  1.28 turns/s  eta 0.1 h


  43,800/44,020  1.28 turns/s  eta 0.0 h


  44,000/44,020  1.28 turns/s  eta 0.0 h


  44,020/44,020  1.28 turns/s  eta 0.0 h

shard 0: 44,020 turns, 44,401 calls this session, 9.57 h


## Step 10. Check the shard

The bands come from the development runs: an error rate under 5 percent, records per turn in a plausible band, both stances present. The records-per-turn band was set at 0.2 to 3.0 before the run. The corpus came in at 0.15 because 94 percent of turns take no position, which Step 12 of T2.2.2 anticipated. The flag below is therefore a note, not a failure.


In [5]:
back = pd.read_parquet(SHARD_FILE)
assert not back.duplicated(["packageId", "turn_idx"]).any(), "turn done twice"

err_rate = (back["errors"] != "").mean()
rec_per_turn = back["n_records"].mean()
all_records = [r for js in back["records_json"] for r in json.loads(js)]
stances = pd.Series([r["stance"] for r in all_records]).value_counts()
lint = sum(1 for r in all_records if r.get("topic_lint"))

print(f"1) turns          : {len(back):,} (no duplicates)")
print(f"2) error rate     : {err_rate:.1%} [{'OK' if err_rate < 0.05 else 'OUT OF BAND'}]")
print(f"3) records        : {len(all_records):,} ({rec_per_turn:.2f}/turn) [{'OK' if 0.2 <= rec_per_turn <= 3.0 else 'OUT OF BAND, inspect'}]")
print(f"4) stance mix     : {stances.to_dict()}")
print(f"5) topic lint     : {lint:,} flagged ({lint / max(len(all_records), 1):.1%}), a review queue, not a removal")
print(f"6) empty-list     : {(back['n_records'] == 0).mean():.0%} of turns")
print(f"7) prompt version : {back['prompt_version'].unique().tolist()}")


1) turns          : 44,020 (no duplicates)
2) error rate     : 1.8% [OK]
3) records        : 6,449 (0.15/turn) [OUT OF BAND — inspect]
4) stance mix     : {'concern': 3308, 'support': 3141}
5) topic lint     : 184 flagged (2.9%) — review queue, not removal
6) empty-list     : 94% of turns
7) prompt version : ['stance-extraction-0.3-dev']


## Step 11. Read the logs of all five shards

This cell needs no GPU. It reads the five call logs and shard files the array wrote and reports the whole run.


In [3]:
shards = pd.concat([pd.read_parquet(SHARD_FOLDER / f"records_{s:02d}.parquet").assign(shard=s) for s in range(5)], ignore_index=True)
calls, seconds, versions = 0, 0.0, set()
for s in range(5):
    with open(STATUS_FOLDER / f"extract_{s:02d}.jsonl") as f:
        for line in f:
            rec = json.loads(line)
            calls += 1
            seconds += rec["seconds"]
            versions.add(rec["prompt_version"])
print(f"turns extracted : {len(shards):,} over 5 shards {shards.groupby('shard').size().tolist()}")
print(f"model calls     : {calls:,} ({calls - len(shards):,} extra window calls for long turns)")
print(f"GPU time        : {seconds / 3600:.1f} hours of generation, {seconds / calls:.2f} s per call")
print(f"records         : {shards['n_records'].sum():,}; turns with none: {(shards['n_records'] == 0).mean():.0%}")
print(f"prompt versions : {sorted(versions)}")


turns extracted : 220,160 over 5 shards [44020, 44034, 44046, 44024, 44036]
model calls     : 222,101 (1,941 extra window calls for long turns)
GPU time        : 41.6 hours of generation, 0.67 s per call
records         : 32,385; turns with none: 94%
prompt versions : ['stance-extraction-0.3-dev']
